# 🚀 TASK 6.6: FINAL LOCKED BENCHMARK TRÊN GOOGLE COLAB PRO (GPU T4 / L4 / A100)
### ĐO ĐẠC 2 CHIỀU: 18 TẬP TEST CLEAN & 6 BIẾN THỂ DEGRADED VỚI DỮ LIỆU DRIVE 5TB

> **Người thực hiện**: Thành viên C *(Training & Evaluation Lead)*  
> **Mục tiêu**: Nạp checkpoint rút gọn `fatformer_crossdomain_weights_only.pth` (2.31 GB) cùng cặp tham số đóng băng $(T=1.0811, T^*=0.3700)$, giải nén dữ liệu từ Google Drive 5TB (`test_benchmark_gans.tar`, `test_benchmark_diffusion.tar`, và `test_degraded.tar`) sang NVMe SSD Colab, thực thi kiểm chuẩn đo đạc 2 chiều (Clean 18 subsets + Degraded 6 variants) trên GPU và tính khoảng tin cậy 95% Bootstrap ($B=1.000$).  
> **Môi trường**: Google Colab GPU (Tesla T4 / L4 / A100 SXM4) + Google Drive 5TB.

## 🛠️ BƯỚC 1: KẾT NỐI GOOGLE DRIVE 5TB & ĐỒNG BỘ MÃ NGUỒN FATFORMER-XLA
> Mount Google Drive, tự động phát hiện thư mục `Fatformer` hoặc `FatFormer_Hub`, clone/sync mã nguồn repo và bung công cụ đánh giá mới nhất.

In [ ]:
# ==============================================================================
# BƯỚC 1: KẾT NỐI GOOGLE DRIVE 5TB & ĐỒNG BỘ MÃ NGUỒN FATFORMER-XLA
# ==============================================================================
from google.colab import drive
import os, sys, glob, shutil, zipfile, base64, zlib

print("[*] Đang kết nối Google Drive 5TB...")
drive.mount('/content/drive')

# Tự động phát hiện thư mục gốc Drive
drive_candidates = [
    "/content/drive/MyDrive/Fatformer",
    "/content/drive/MyDrive/FatFormer_Hub",
    "/content/drive/MyDrive"
]
DRIVE_ROOT = None
for dc in drive_candidates:
    if os.path.exists(os.path.join(dc, "datasets")):
        DRIVE_ROOT = dc
        break
    if os.path.exists(dc) and ("Fatformer" in dc or "FatFormer_Hub" in dc):
        DRIVE_ROOT = dc
        break

if not DRIVE_ROOT:
    DRIVE_ROOT = drive_candidates[0]
print(f"[✓] Đã chọn thư mục làm việc Drive: {DRIVE_ROOT}")

# Đồng bộ repo
REPO_DIR = "/content/fatformer-xla"
if not os.path.exists(REPO_DIR):
    print("[*] Đang clone repository từ GitHub...")
    !git clone https://github.com/LeeVHoangtk3/fatformer-xla.git {REPO_DIR}
else:
    print("[*] Thư mục repo đã tồn tại tại /content/fatformer-xla")

# Bung công cụ đánh giá mới nhất tự động (Offline Self-Contained)
os.makedirs(os.path.join(REPO_DIR, "tools"), exist_ok=True)

eval_tool_path = os.path.join(REPO_DIR, "tools", "eval_locked_benchmark.py")
b64_eval_data = "eNrVPV1vG0ly7/oVHS58nrFJiqQt20scD5Al2dZZlrySvLhAIQaj4ZCcEzkzOzOUrdMJCC4/IBfcPeQhAZI7HJIAecjDPXmB5GGDPOY/KL8kVdXfM0NS9u3lw4DFYU91dXV1dXV1VXXzC9Z60GJBMoriSZ8tinHrGZZsNBqNjRd+8SLJ5mHW+snBdp+9jsI525ku/Ji9niY+O0nwWzxhO4skYD9gu0kEBVG4YK/8BTucTBHeOfXzC/ak/cTdaLFDP2VFCP+zBKrlCcsWBZskcZ+N/WJMTXlBluT5KJn7Uey9D6PJtMi9JJ5dtdNiCih2sea5D38CRDb154iHWg2INNHe4z47ZQPWbXeedbtNdvoAvnTaj552OojEj6dsEvmsB5WwapFAzVEUxv0NBv+6bdGRbp91n7GCyM4LtjMLAc55xl5uH+bsIet22G40Hi/yKIldqtmTNXt99oSdA0agMWS74STzR+GIOT9Nw4n3zaNOk/GnLfX0FJ7OZ4vMy7vyoddko+R97C1S5N6rBBnHvlmEBRuF8HnFRj4QB7xw3ixmRdTamfl53mdplkz8uMny4moWWk+AMLgK6NHAGPh52IriPIzzqIguQ3bpM38W+Tlz3h7uvmmyr7/yVD+b7PIbb6R73WKnEXCTODgHcjL4zHFYIxiaUTgrfOD89s6Op3jQoq/ES6v2BQgV9KWIYqDoin25dY89T5IiB5Qpc54PYCw7HRdGDCkOcORw6IOkYLMk0r1ptXCsvNFiPr8CIMB0gYJYTBcAN4mg4iJg54nAI0UA6v9k4RdctGQ/LoDT3yx8tnPyNfLkxydHhwCA1RKaIBvRPE2ygiW5fMqv1GMRzUP5/NM8ieXzZJacy2doBERdfvOzSepnebgxzhIg+CoNcyZencDnLDz052Ge+oGGgFkrQXajoGiygyiHv6cLgG6y7fiqyY7SAgbKnylq48U8vWIwunGqaE2yYGp9accxgcTl0vZ4EQccIwK8EKTQ20URzfL2yIcxl0TBcx4CRfhwkMD4Z0aFywhlSMIiy3PUAjkHebt/IF/tz/1JuLHx9vjox3s7p97x0RFO7iRvp34xbfvnOX468vtPkyhWX0ZRFgPbHM8bR7PQ89wma7TbDdfdiMbMQhiDJIHswRBSTa4K5Lc2zo+scGCSmpXcDUQz9XO/KDIHgfNilCygw40sDJJ4HE0WWdhwObIiu+IPEjMHbhugThhzZTxokDJucL0SfgjCtGB79AE802hSmPI1NIRZdncaAPgPp2EDlKJ38u75yd7pCQzOGb1tcF3UAFqkDjKfe/jlPJpQMa8g9RMHgynBHyf+QjyNwjAd+xdhY2O4sbG7/+LFu5P9o5qWJ4sItA3WmI3mXq/TMR69YDyRX7v4RlSZQRVvq+P1nlKb9BXew3/7O77nVUb+bBbiyzQezfHz8hulHInC7YMDr/vMIM9k00NW6QCvsbv38nh7d2/X+3r7eH/70OyXXEGwMbmG6Oen9CzWEf1IjBZrCZEl8EJDJzvH+29PgQJs47rURp81DmEN+/HbvZcsh1X9q8GjDnO2syIa+0HBnn145kpOKFrMOkWGBsI56vivBlsdG/apDRvDQvnV4KkCkp0AmJf+Is8j0MjPoYw5eTSZ+7geuDZsbylsz4CVbADYXXhsvUvZ8yhYnMPS4PR6j1nrR6zb7eEHfHMbGzcbG3xsvO2D/e2TPYNPQrr77MwQ9LdZAmNMMpElHhYORctqCmAFcz6c4LOoROW11XpWvZ5ZsWfV7JlzzOspRGKmIRY56VjjeTQRLUOZ1a6ailjBnJc7+CwqUXmJXD5rObFqBp/Ao+qin1lVxOzGGnqiw0AKeCizwJUKwArqCwDuwvML8YzlnnzxAhZNsGXRwgly3SxXEdSs0hYv1RMv08YOlm3vvsEPHya7xCLVC6IxVM3B7htD6+hCfCpVJXVkVJfqSaAwtBV80+/kKxNb1yCkqwnpKkJ0YdcgxNR8xA5bE76sUYxbHfhi1xeqUiPQuvNlnS6Fr91ODQ6LCKlvTRwGGfDVoIOrYxILqZh3QZ/ukUBAiRfKp9CYF6S7aRILJY5mr1TqxvhLeFPHYzXzO9QyzWW+JHil96XXrdLrln4/RPWzMQrHbBzFI7BhMrBtww9g5cH67ARgP0Zgb4Vg9aPldyaNvTOwmodDF5WYVcTXbDCyWMDIzlb11WIOtgRaQoEuwX9gHECLi1AVBh6YXTVGWOCamORbojh3qJJrY87CYpHFHJ96MfeLYBoifjSX2/jHCZoAGyyyHLYog9NsEVotiRq1uMs0CtizzpCjEGCHSRwKZs+SALji5YtzsF5h9DLnHOxYL4P9SB92U2BbiVcX4RUVLGM1bhPw8zSa633IfIG7EF9vLGGPmMBmDlbJ3A+mYvMFO44p39LAzghW0XQKixpsKQuGu2/Ag4VOefPmtq12xWgq6hkMPRaUBka9NwbH4goWEFk0KGd+e5a8DzPHJVnyUZbsNbI9CQtH86jJzvSXoTvkCL/AnTa5FcD2R1shgF0T+AcS2tx3kQP40GMOcmkTdsNyXnhUwEcvms/DUYTDpcUZaVR9GlYN39o6D6WNJf9ZGwqFDsjg3R5htwFmBiy0RMS1sBjTIMoRbjlaXXF4J5NbTWbZCaSorm+6osUGWT8qYAzszqi6pelKVRagSVNj8lNPVI0m4XMr1SqM4HhcBhWpihIqIEUIW7VxQzR5fQWxnFs2x75AN81XhhuF3A/gEpBzU42JYrAYINC+TeYJTr33Zxe1E0fJBsL3y7IwulMvS5qLGCylZKNGaQXo+mG76PR47gcX51DqwGb9TTJazOQQgk6Q78j7NYO5hV6Qkn+EaxsYTfby7btNWaMtNQqqR8+L4qjwPCcPZ+MmehQ8IiDMBz2DEfkixY62FbiWCazYhkXlEqQI6NyBp97IedRk3SdN9gg9Vhks7gNwWKX+iPai3VLlccCrHkRx6GdO9wl7QHY7/bVoEgyjRTTJ3vvZSND9waB1CthewC54tnAUbc4HY0YiwLR9GYXvnWk7j34WOh3wJbQMssSICOqcqavGhfwXuEALf4gjPvXAiALw9wXmQoC+sSmqxRCVfiCHbdVg4ESS5gD6hM5ouQJX0HAIL3PuGOuDuCcz7LI/y80pTsQTCnhJn6VXAgG+FY82gPLiAIT26MAAgycnDx1bwRoAxyHxFHdgTdp4uc1loKfJKax2CUjWUpBD+AtzCxDOwc04OOu0Hz/rPt56/ASkC563nj7rPd2ix86zXrfz9NGwWatorH+gGBFT78mzJ70vtx5j9d6T7qNOb+sZPT/devr0abc7NMgaGrLnebMwFsNUXWbhnaOZb1WDpRQL9QiPTMEl3T0wBu4M3g/V65l/Hs7EWMK2AL81hqbhZA2rrYg+4BiSZw8dlTFOTzU8WuuCAPVL43BVVWjRfALYaCK0kxQ6ywkipxyquMbQpTmHTrbG8cvnjer68UF2Uw20A1htuNVrgN2pn4UQa6jvlBiUD03OP2EVBiDEC1hXwZ/hZ1fePAQlFeTe+6iYwlo7i84dQoBqtIApyFvhsspFAnfD4ZJX4RyUpQ/NhhAOmSXgi1bhCw4wzcJ8msxG+rUIaJDtqac6vS2Zn690gCSeRqRnID4C9hUZlVAh4jZoPCGr8vRBxYpE+RTku2wAbVck+NpidWPuQyjH84MAtkidNvhNGwkMLu7BzLIs9M0CGwXu3i1of4HPW+JLKp5VpRs+a3IYC9i382EANomHTZPHBAj+GdrDvGjnybiY+x8cqyou+POB1PBEjKxBn2d9WK6G7SBdwBpHXnWxwAk+eTHaSOJLCUy0H44Qm2Pg/tFAj7Tb9nMMAzhRXIgqReohy3DpS9v5Yu44AgmMiAtBOMdsG8s4RXGpVhlKWBR8LJAi2c4mQ7bw6tBb18X1tYNMl+Qg7XXkdGvI6Spy7FplKM1wTQ7VkOTglxpylMzR5NiCl47q1EOFkKM3pBFbEJQo8jVJrmjVmAFYWtP8F+z4aKe1/W4HZ9b2W9FVWPewp0SzKIrDCRUhcWqbxgGBcNqjcRhrnoH0N5mPMsVngJJ9WwUnGTj0OW+BWlAwhSFfhorz4wsOBbOiuPJm0UXoUF0b5ozKhhIhxMdCYoWBU1eALnhYCXtMlUsjq1eehR59VQkUmWADDBx/aKEcbYLB3jHZgAO2gGJHQhO7ysPBV9A88Go40qpjCb6B2S9oZqYQnGlEug9FmnOswWKOHSnXV6JM4rwWuGPwETwdFBwjKlJUXg5+PEQ0GoyEwRBdWQfYV0eLq+YPcK1GfiuK3FLitK44qsSwckqanQMaZSaoofA5nCwwgQzFz4HU1DWA+GLA3yNbZ1Hq0BShyUHdci34tAqe1kHflJd7GYX2guh/ZYVvCrWhCOnD3pFqd+TLPAxHsvRxr2wSUDz4jJBK+6Din4KtxtoIfE0A3jYUgLE8pt1Gghz8IzV+7mMIm9Zjw5TQ2k++/yHbqpoWYmU/Rxkg587w/9dKjy4B8htw/WkMpWHJY5kHxjtXE4KPwO0owBqCP7A1hm3NwPiehekMwhsllyi421QfdIfOZBtDC7BW42lQUy1xg0VjX26vaC25xmSxKNA2S53dUgHVkiIW8mWGy4ZJz2qbpdRI16LHrloB1epe01NruWh65iWDRVorpO+UglZCQlOg7aewhxo5c27P2JMDacM1jhdwkoIIRP09ygIpQP7qDMbJAeS9LVpCtUwOVaUpZGDV1/ryaU2tdcsIaFDvyy3Qw+DMWgD9nCzYfIE5pUqwTSxyLW18vohmOF+DC5i28wRSixxw4gcXKbjGCtpCCs88anbz+yi8hBkkNTL/tsYTAuoMUtxgTwTJPnNMLwPleInfZeIaJF6J3DVWTlJrG5um6rY6zZB7jbMHQ6ZasLx2XL1CKHykHXNtTFgp68SSr89ywbWLxOEdlfoHE2ryLGgT41RaEecplXH7GZ88tJGAIaWcI0eLoGh00Ng52H/b/zo6bR1sdh839FqrhmCgnvRLi1RdvABHap7NSY3ZpRMfg12lFwDqQe6Mh4q724ENPL1xNzSTx8TlXVzRLpDVI0xFuKCUPIw0WNmNzDk5fsMetV6HWYz8echeUqOuZj3xBthi8MzR/BJ8hjEvCSU5t8thMBvELYuHQXlMPloJTis0yF+fXZdw3BgCUvG/QLy9CJXbA6byqExCE7RT6lHUCwzIQQOWMAhFmqI94POj7Gw/BXW/l2VJ9n20aAX0wJSNAU0MkkfYcI0OCh4saHDGU3mDkrbwqZ4E+jyzahiLnwC6di7OnvaH2O5FG9MTihy9Og5WAwc6yABtstiF22eXtJpfQCKkapk73hz3xpikOIWoUeo+b9nDLsjuoIs7KMp8nUeQvRJPvAC0YSGMJVheKLcrw+W+ISEgkpbDIJ0NjfVpEYcf0jBAy385AgOoDoeSwf/6m18Nec7uFB3R4J2DFOE3vPXBtUXoTZO9U1gH12UypHTaG1XV0J8M2Q62ALmVfVSLZIOi5vOvTOEvwEtVkXtMuAUwSztPaZL7EU5da+7y8TCUo3rZDi/9mWOFg7la5IsPvl3QAHI/vccHXNfvMxVt4ZpolRte+HZrliW+kGN42kMzT9n5T8QWAFTn+yS7CCGuJF4J+79+Oft+XIt891B2MDbtLU7pW50T8gRCbTN0QRaQiANsKygAPk1SUAULdJlgFBjSj43sXlhndW53bWSb87m6YZAEkXODQjXl72rbK8ZUuqfNOA1h18bCQD64Yhvoc8eCzm51RjLjVY/iQD/CrJ8uxuNZyKd90xzRgfEs5BbNS7W5kXsey14/E9Z50UFtCynHbfwjBBlVmEzchUw0SL92SmFKcHBfoRbjPal6/z+UJ4v8Z6CGfUjbXxSgyfNCwLZxF9BkYeyfwyZrYJaimdwIFiMfUnCrznlY0eU0heCbrc8VL6Txi6mzfK/FzVO3WkFuyUSNK8HXEGJoOaSiWywDX1PB7WxYkzTbeSehyNEU8P0i7F4wfqLHiJYIW8wUPmOLZSEU5WWMErwOJdc6PPgA+O4YlbA71jSJ0laVoSoGxrMBIFXFQD0ZhpfUnJyE5tL25AAI7Qp78Vjm2kQ4BXNHZ94IKx52T3LLq1Vfye1B2qpO49p6iEL/MuZPIVZaOGT0nzctD5lAFHa0uDJyBUA9Bj7pKDp6IWpx3aQOVOhzJKDZyNzEQAssnxz5Zof2pZsP2mk8kblEm13aHPJCdZpE1gdNOANN6czppAfZz65CR189zHOvx2y8rzRS0qea8UtShTRANYgptRH59DC0Z+gtckIYZaXcCg/WDHppJFfUNpWicYYoUisHKRUVidQYDHvK9KLMjHaezqIC027yMHUNdwZoc+Fe8OMrhzA4Dc7AhohN4WenYTRBzbsmCuERMFB0Vd6p/OzWoFA4vmDvFpj4FAtDi2JyUhonFLIDoQOxWtAuBmV1hKW4LvIzLLbdzDtFSTWcumrCD9GRheCihVnocIL6dQk7q5hSn5ujOUJLXC3MOWC5qLwJZ7rBWhYubVCM4roGyzmOjqyKKVCcbHdJ2qO1ao5jJJLEtZLeM46lWLZhxeG7CKeBsw070f5pKj9D/vA+PE9rl8JxJcGKT5RxXJtZJbqyLF9Kzke5El4bkXew+lJMDqbcAAi13rj1Q1MrTYpcNb3v0ET3Ru+XDcXOfgTBL5RbCjBpol3Yj6uQEy9xAdZcErQmiic89oTO02P6MNzQlMjjz8ZoZBjtbm4ac4gXjqRQAcI2LzIogvUIBqREZZNQG1aIxCTmg4FJ90RjMnpnLXdgl1ByiEGWazQCyz+s/GTMWJQ/tJo3+dMWNqgjq1Z8S/KF5c8zlPpDY7jFGo4HNqV1SafY0C6WJ9ra29lkMQ/j4i29cTCYlkUp3/nbjpgXEZ4sOyA3H3sOJ5Gmcz+7MM6Riq3kF7DvUzvDHzDpCDPab0PiGPpmqGGn0WrprSPMPLRFB8I/OPZhUR1ApvgppK6ebmq4fPNuB1MFTcvalT6w+mbJf9bqPgZEqm+i/2S+ZeQhQeSwfhegsRUzHq9s1rDfZMMiAiSblqdjp+EsHTQOdWLISJ2ylS2t7qGyBZc0JMJYsiGeZAIGIZ7LXdrWF/L0YL66l5AQj5qxnrkN2eipXE+TgHaT1YO9q7s4EsdXP6utmqPAq1sTU9+DAfRUy9wckm2DeOq2tzqd5QlsnCg4ro3b7AJtCB+2WJE0dEfqcPIL2MG19i4xYAIee9iRgDGCmEErz3zKtnXvyKRLP4MjWEVezygICzfWkXsfgO7zpEedD6m4GKNZRMb5OfVngUbTlRKcY/A5wZ4O9MLRooDN0SqaE4JAI3O1VoB9TL6693qbXztG3Scraxu7/9rqndW1dRymvnEM3a6dR+TbgPp+wFVzDjvP0IP9R6hke0c45VBR8ExQO314nYCgI6Ce0eQUQItAbJDhK+Yl+pd+NEM/giN8sMJVzI8k8FCFaI0+yB2vM6wo3DJoQKjqyydmdKDxYvv0xdHxm71jfrvBi/3D7QN2cLTzem+XPd873Hn1Zvv4NUjQ26OT09arox3Wa+3uv9k7xMOaAPn13vH+i/2dbTw5CQp5++S1uTpJ1+Zp9YICmHzoxYf7CbR+prsKrpHwtumsaz8e37Cfs1OpXfk1BgJO+ewQqtIsqvLzqC+AOdsR1w4fOZn+7VAUye2rxqUQ2Bg1Aw3XpXJl8K+O0ZIr86F2KXIWmV5cPNQBC6KwLPJkdole4osUt/B1Z5x0xjA1oDEZqR1/wMq9Bsv88aZZnUcRzpFN88fl+psX/mQyCzejGBTK5oMHn0aCOoskWQJMuiNLqjE2y6gwiiHSDSoCTbvNJRCSCSshhyW/ek1s1hrapt0tGZM1HKsl8VMCNPi+/kmj8dX+3ju812N3+/AVe7m/jXbA6fZbdrp3csp2Dva2D/8oLYt59GdxA4znWn0ElN1+++dE27//8gj+3378ux2i7vbjP5v0GdrJEReRqFPk7qppG6CV4+FB/2yUe+dXwqBAlXIjgwgcRqb4l0ZFeoR08YojZTaQ3u/ZjayOjI9gUUHNiYZL9SKWFE9GXC70AgS6rcDokR0vxx079BS37PZh/NJWv547eG5tyMonw/BfaX/boP6Ivl1DrRvvOrqRG3+5843YPda7qW720QOh84Med+wt+HBV9EyGiuliGIybVTl1TkFvit9d2+Ny81m8Qg81eggHNQclbfzkgqych5P1zdg4PxIm3rh1J6LI213vK8Y61qZ5UOJg9ZSCgfNs+CmSgJWMbKKkgG0xrzLGDBakETKDcMt+yZ1+FCdegrSNgUXog2uOAWARr4m6CyvgvAyRCEHLXP1L9JB0hpW48veAFxOvhkvCxacyblvQFg//XFdY1G/eCL87yOU1+T7MPrs3m4YIg8hab2/sNIFSv0pn/qwIc6CnBe3KVIwZaIFWSphuGq7lMKwZaCs729RhqwLaWB77SSnogFRhKBtCDZBTQKEH08Yuqzp71tbqbkyXqfh97zC3P0sXft/68A46ketFsVir0DK/VsrsQw6bC2OS82JMTC4HVvWbanj1DrxDLSQPzi2bS+JQNCVclJ3Rqv46D/Sdo2tAyKq8BYUP7bUB/bU35QQ7UHTZL7kFNzDzFnR+mIp802JghL8tODP2TYBGQbMs3Cr+bkh56fifEbssb6VKkCqIaW+lFJAx88vyIr3bmvluDWwp7myMjoF6AhmPKUXlYKQaaMo1xOKIcmbeE8T3vfpWCDPNCJbZ9+pGGmXGE26YX7qN0tkq4UyiEGLpFQ6DWEbhPapnk/7m8mNeQjLPjMLSuUoz/V9Bq7Lh8vNfCliVlYH5cQAFh18rIKkFkVYAxMTx8hBx8YRUUYYJqcZBs7KmJC0jh1uMSjWlimGA+uxaD0r/y/xmyFA79rtb+Q1zrsvs7v9oi6+WLuze3yBf8fo8XK5EP+4rZt8f9tu98c09AGyUVOW4cYzBIaOWZLpRCW/OMUEkq1fihdNOZh1guwG+/dZ6l6pXzrVgK33P3YZx4BWc3oGHx3Yh4PUetBnnFi0Y6toN+uZWLi3BCrU3gBzK0MDy84r2jOHNLZ8yioiVcwcNwezMKhzymDeta0DtqumkzqoQM84ya17ZaIbl49LVczYmHj3j1qGpnMQx0ei5uA5N6ayOwkCzdG3ltL5uur5q3YQWg2K+qgyKPdnpETzONKvJLMh0BdvOAEEE3FyWhpTrRHqdWxh4fcnnoDCvA+IWAIDzSkgVGhxqzigym6Jp+HwDXgMPvpzIVGZdH0my6isam2a7EolyNFQxifNeFjKrY4DjiMModEfgRQV7qmE5KOTBsSVZWOUF2cycqhh3SzOyKi3V5WdV1vP6ttbmanFQOk8TRGbGVuVg2SouNJdTXZ/AtdwIWmYAGUn6mjhhm+kCI+GLUwRnNshGV5av6CrZu5XDH03m0Jk7OnFXN/D8OExVtGz9h3u/qvTRGKjMTtvr1jK9bnZ1nDX1rhXuckNp5bf+8hVazTLYqz5zea2+Xp7RHithr1+n8axwHWx5Ga0HMtfTyltj0QEwYUM06jqPU35t55UuEBxQegJ36zAjlvceIe/aew67pvcCaEnv+du7915IzloGCF0lTTRTs5HDwl0pAlKEa/jg4DnLnX24SO7ank5wPxkHabLym654M3SXGWXL2l/DWw23hL0KYBWHxbxrWe7nP5Yzv2c485+w5/t7h+z01R6TV6gy5+vtU3bwp+4fhQidFlB2NICZMcd8XNj9i2i4drSDsl1AwAR35yq8j2EUSN2XYTQj26DuCFEVClKi8Co222OvvO51LVZ873XRiZoIRc+KUCDLbz/+KzL99ttf8DiF5j0AffvX7DnUO2RfQyADRuK7v8XL4vcOTrfZ7vHRW7dRacgOW4ge2D2WCQZkH6FJYZv98rVXJNRnvIO87lrdFbfY1KA4u2zjoZ3UMfy69WSJlNNGs0G2gqpnJ59CtTuFWmxpWBFusQXCTloE0vA0b5MeyOkA1Icwi9E6CJ1yd90KNzzM2gJK664OppVeItZNuLXe2D+Lz64FNXiJ/s0madQKAbAx/vdf0um96Pbbv5ijz/b24z/hFYi3H/8N00CgtM/uX8vG2ouUrhu7uQ8aS9J7I07wVTojnHN6Ttq5lGpoqs5RrvRMR/qSW671nbL8MtJhbdhHuHtsrPU5lgbdq12xy92x2C/FsZvP8svW+GZ7Nb5ZHbdaHYP5QsQMxNV7MsNnKlPpcdzlbZd0Jaa/UccXHo6y8mWrU2KZZBouBDscJTAvyTzWt5QuH4ZU51vrq0zXktl48KBhELv85tO1dyymdxmZdfeo1jNbX6O68X1wUzvJMH7yevrd7zE+8t0/o3DcfvwdhJSm//EvbH777e8COiRt6gFDDcD0L24//j0cFRTNwRFBWH/+En/Jod2o50bVyf4ZQZN1gVFBTk1E9A+MjK6PkK68206GT1cCmaFVEtgVGYqrMWEy9uBxbynMStnGLvZXoq/RkXbAVgJht6UvZlgXKBNXKeLQG0hlPLRKxZwOsfwBgZi1ARk7MFMTlLlDcOZTgzSfHqypC9qo5bQe9O6hmzuHcJaL0/cQdlEzrib0onyPwqCBxVQZRisByV4R0Pi4BHpdZOcOEZ7PivTUR3xWxHpqYz7Loz21UZ/l8Z5y3Kc24lOK/NTFfD459lONAdVpFRUKMsNAenram0YLeKNsHEG+wDzkV3+TaVS2Vs1mq1ppUudwtvTfSpe1RfZnoCq7rm3SqF+DuwY6eFfAyY/tiH4pf1+V2E9DPjKRj9YgTz4RucmkoVsViGw5vrpQzTp04+Xo6kI269DBxFqOsBTBWYsrXYEqXYWpZhbBj4t5oyyhg3IcbavqSq5WrHpmzqSGHv6Pq3NSwXzTJ1Sw2XH3Top7dZjxLvjIn811Ltd8fG7SxUz1Ncj/a9YYrathXxLIKyXrKhlrB6+RrathLB+8xnhdDb6CcGAp66vAUw2drmWTElLNKFV0x3WllD0gxa//DHMHnOuKzFD6nMylAyeviLqJvAHOcuna/c9fQ5GmqP+Qyuu8ytq7jMsD1JqYiH5OAQLEZZeKtIPMLhWZBmOjdMUNMegwoh2h+BWMgl/wUjqTBbu8hH6tbyR/3ZH/tiP8eoQ8c+UqT9D374v+yTvwMr/YP9gD/xh7vn3EduD/6dHhS/bq6O0fpU3YHs6Bj/gjA9yHoE8ywc4DvYJeciHcBfrnNuhXDuUPHPLsYPh5QzOYl1/WOlJM7OBaglOaeIso5FsHcB5CHNP0wCWQRR/agEOMJ/3mzge8bIzHD3BnDJSV4idrI3JlNOr+O7vC+ujWHRBhhTsFiu6AS9QRAzAN8ZaVXHFCmkpNZaw3bdXeNK1t87bspqHnxL3Y/Ebspm3IDvU9MHTpuhpigHvfwBtaSr/0h7/paLBr3H6fwT4THdlcFKwu4LlsDBrY+eMZ/2ELmzelKwjKaHO8nOpsOuSO9SliWNGS4UimpONdn30wxZqyZPlpKy7hmd9XIbT88kZrgV5pQqgfKcWf/OQXLEhJKBvtfeOO4cmnzBpxTEbhq8wZU1RGdFO07V5UpkizZGw02VqBWpZt9DnCtVF1JmpJE0y5o5ytlDVAVZU0S9qWj81qiaOwBVxF14AbGKqSV9uune6+QvKUKJHwCXYYovdIiB566a2fluWp83AtljgPRpMIf0H2EwVMa2WOo404RE/wEUIDV3jvk2XyNiBbz0cfFZgrpQQ58Ftd4I8mypP4y47q/wBPSsLp2zjnvxK7x11flEZUwghQcBneHG0juogJBmVMlzE17v1p6968dW/E7r3q33vTvwepS6XKxmn+vuIL/hwP/dyrdfzM5a5D86whd+0cbm5XaNKeJ08ev6ffCFztl2ooD1RdrWp+Df89RpX/g3f4RugwzmUdI99m2aSFZBrICMBdg15RZYZAmVvSGUen8zC+Yx8X4Beg6HjkkogxZ5tdV+fJGacI+ftQj3xFmsQmrPJCCKFY0Clf11rim0ugzV/kKy3my6qIxdmuIAqtKje1iaBInGXG1HJCqaY7MMPYkfJdXalwKRll/acpWT2SKi1W5Dhq9W2onTuqcARtg0Skjqlc4L4a8KPHIzj/jT/mBEoBJ5afB1Fk3nxZr1WlQiTtKLUi16cGfYZOfdzGXzpBL1nEr5RQlQL6pfoAf6le/uL33Y5lPr/9+Bsw4k9vv/012vK33/7mLXsNeRC/eAPpJ+9uP/7jIXv96rtfbbMTgMAigNp5B0kPO8w6Yb7sjLjd6CenJq3PSvqsnKRqStH/9Sylhj5hWp04yzJdWvWZLuPG9f3nRnTxfv+H3c4Nu77/5rvfY3Dxt9DElYg/wrtH9I5nB95zFTBuiK0CuQ8XZT0sw/0xDo0Cwq2xVfCfv0bR+h1cNg3t/XbOa1ZzZipdqW6nKKhYzv0tlVVEzuYKmdOaHbff/hUcXctuP/4D3V2Gpg8yZSGZsjIHEpDgyK3JFayDqhVcYMtyQOO0w0qExpEHBXcfPMAdcJVQb90781/k3EC4l9I7KjJZH0hUrAYowWZgm2nlI2mPRLl025nkQrH0zZWKDQecya+8nj15PTcQvXIXwYuHpvdmGT8qeV34i/UeebE8j+IEnkd3I3gij4vfJrXx3/dxehI="
with open(eval_tool_path, "w", encoding="utf-8") as f:
    f.write(zlib.decompress(base64.b64decode(b64_eval_data.encode("ascii"))).decode("utf-8"))
print(f"[✓] Đã nạp script kiểm chuẩn 2 chiều: {eval_tool_path} ({os.path.getsize(eval_tool_path):,} bytes)")

plot_tool_path = os.path.join(REPO_DIR, "tools", "plot_three_way_benchmark.py")
b64_plot_data = "eNqNWMtu48gV3esrLtgwQE5LHJJ6tCRAC1vuTidpP2JrJgEcgSiRJZExVUWQRbc0g1n1YpbJYJBFkCwyCbJIgCyCLALYiyw8P6J8SW4VH6JkudG0JJJV91W3bp1T5RfQ+qwFHvdDthhCJuatvmxpaJrWeEPEG54sadL61bvjIXxJ4SSkGZxy/IQwDuRLG04oA30SJJS2fknWMObLmCREhHdUdnnBkiS3BkqQ9BZ6Zs9oXHNICQtAJJkHfkjZsGGbcEJSGoWMwk+4B+MvL6/AsZwO6Ge20XBMOOMQhKj0BkVEhmKXEWFwXNjtmjZ8DmeO0WhvRU8JnKF1mHCUPJVP+lkY0VRwVO9J+Q68IVE0I96toQbcCJcxTwTwtHxK19Ujy5bxGkgKLC6blkTEERdRODPjtXyS3XEkGo3Lq4ufvR5P3KuLiwmM0KAZExGYZJbKu16+/4aHrHrxw4SRJdVdd45huq7RBM00NcNoXHwxccdvj68m7uXx5G3NntKv+0KV0+PJ8fXriYaPEV+k8i4wRy7m3hVymtz3ZO16aprClDMzZgvNwLHPISApESLRcdRmKnyeCVROqMfZPFxkCdWMYQPw2vabtV4dZ1vV0UhTdSSNNnw6xzRhlIUqZgdV1xE1s5TqWkrJjCesdWe5/db7IBR0kYS+BhjM832sZoXckTAis4gCjVIKGvojWSTQt/SGgTVBJyu7CWTlGJg5pZnN5GSlOjY7TSmUhl/RkW73mlihXcy7H4ejtmXhAKSVF4D1eZ0rgT3EUk2w+oksj6KUu+AFoXzzUCLioVLzCEbMk5Cm6PdGNclLG0cUC/KMeAmH4/H410yfbB5+zxYggs3DB0Nr7ou+Ibe0kDwPNvf/ZLhqNg/fyqKez7M05OyA0hUlUaF0srn/KwfBH//MYLl5+C0wyg9pXIxbx19IhSs5pgX6+GMIcfD4Fwbi8Qcv2NH58Xebhz/ABCdjjCGt4fX4NWqO0YcayP3fYvDyl83Dd6JUneYZXdpu6vEkz0zXMi2rCfmvbeX3rm32cTo6fdN+NVVzgA4g9UhEfVihFPgUEGK8IFMOCf76HI5y807NfH9gOmip75gOTvagZ7bR/KBr9nH2u2a3N81VOjWVQS6Lmm0UGnTMQU9p9rtYMOZgUIxihbIsNhHt2ILqEWX6dsqNvADfh74IUMwynW6uhAtGpDY2YVWaM5LoK2jlYs1tWpplC5Y2jUbaFg8rmJTAiGvb4xFPRtqLvif/sIH6C1o2dtVVrAXl2Kk7bm4Tte+wQNcKbfdgtua4bfedmb/n2Op37YFXd9zeGfHLasSdZwLY4R6J1PrL0zAVYRTByzFnaT0E2uvZlr0XwgzXr2Vp1RL+KcOKJkg6IX6pXEJY47hcc5zgSR6mBJebfIqaRcaKe3s6rIq/FJfSqnPbJS854bLZXFDhBjRcBEI3diRkJghjXGC96HPt62Bo2vNvjmrr68m1Wo/0yuhKNzCH1atKHTZ9LuEsMD5qRdCVGOm4BNofk5NSHueJn440Pp+nVECMXCPSj4UYkJHmUSZognNxhy8zLgRf4suco6rE2FdmN397r9IiRSK/nCSZFfTkilBEyA0SYb5T+4zN/X8zmCSbh395iHkK+nDf8fgPNtyuh7u03BLgE9bLz4PHfxO4RmSt+7edA+6bEBMfu4ydKFYi9G5TfXWgVdVoWlvtNQ+DZ0ZYN7JWBnQNcf8/8A7HA/qRsROm/Qk2wqWcRtvubjsiuqDM1yPuIQfHMU0gonNRtzzA5wR3GSSKcbosc1AtEGfLcc4QrohfsdxMbvTkRg1mGW7OWIDbPCYBN8q8wrWDG4Elv6N6GYsEGiRWk/i+W/BtQbf4iXlEktEkyWjhPZHeXMznx6myzj8V91X0+CyjPe04vqy3Kaq8RBCiyR1uXDnbUZDbGx+3NgiCdgtJaEtl8vccA5bIvx1BkQG2iHIyYbgs5xEnQj834DNw8IukEYcKRZiCEMUf58a0rvlyVDzdDO1pOUfnCIkkwnmEJUUk81JkdbBaSId5Gt2l/YRRCybtOorxSn7VbaQdy+z07VcyKmyeliacA6yZ0yVSoNOvOLQwYXW7vX0TnQMsmtOnvDkVp5YmnMGgMrEdCaZA3lUCqtjyRqfe2CkaO9tUyYpUNZfnsKkMIbkgUiiwHNlymcpXtY0cacNDXPqEfRXpGs96cHY99Hc8tFqHWHOXb3XJrc/b79TtO/v2DzFiYX4HD0GvTj01X3jmiPZ8PbFWgoaEykpRIXZABXFzotDz6kYuMg7I4CEHWQvXmN6yd/sLvC2LvmVPD/QXyLtdbZ+MvLmRde7kxsH67eC3h99+vu2cHpAs3N1ojnUkD1Kd/NbLb/38hrpH2nSbrp66sEdF1d8zW8K21d1LT056J0hvHzKQ3Pd9AcInhMMvsscfBJz/+AFR953iwWuSwVvJhRmidLa5/zv7VJ6rRfSULxIpj5KzGV+5grsET/A4Kt022xi0aRufwCXyjCWkHTciazwkFqSAh9YlQjVWQPrkzLt7wkUndIW7PZffFixRHR3JHUVa2ZOvDmxF3CEGTXHnooLQttpexNOSouIEtzO497r535++n8r/FawyUlIdx08oj3TqHxwzKiF6CF/vOv1GnW/xoOq6cgiuC6MRaK4rT7uuq+Vbwvzo2/g/FeciVw=="
with open(plot_tool_path, "w", encoding="utf-8") as f:
    f.write(zlib.decompress(base64.b64decode(b64_plot_data.encode("ascii"))).decode("utf-8"))
print(f"[✓] Đã nạp script vẽ biểu đồ: {plot_tool_path} ({os.path.getsize(plot_tool_path):,} bytes)")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Cài đặt các thư viện cần thiết
!pip install -q ftfy regex tqdm tabulate matplotlib seaborn pandas
print("[✓] Hoàn tất Bước 1: Môi trường sẵn sàng trên Colab!")



## 🔍 BƯỚC 2: KIỂM TRA GPU & ĐỊNH VỊ CHECKPOINT M4, CLIP BACKBONE
> Xác thực GPU phần cứng, nạp checkpoint `fatformer_crossdomain_weights_only.pth` và `ViT-L-14.pt`.

In [ ]:
# ==============================================================================
# BƯỚC 2: KIỂM TRA GPU & ĐỊNH VỊ CHECKPOINT VÀ CLIP
# ==============================================================================
import torch

assert torch.cuda.is_available(), 'LỖI: Vui lòng bật GPU trong Runtime -> Change runtime type -> T4/L4/A100 GPU!'
gpu_name = torch.cuda.get_device_name(0)
gpu_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'[✓] GPU khả dụng: {gpu_name} ({gpu_vram:.2f} GB VRAM) - Sẵn sàng cho Locked Benchmark!')

def find_file(patterns):
    for pat in patterns:
        matches = glob.glob(pat, recursive=True)
        if matches:
            return os.path.abspath(matches[0])
    return None

# 1. Tìm checkpoint M4
ckpt_path = find_file([
    f'{DRIVE_ROOT}/**/fatformer_crossdomain_weights_only.pth',
    f'{DRIVE_ROOT}/**/crossdomain_model_best_m4.pth',
    '/content/fatformer_crossdomain_weights_only.pth',
    '/content/drive/MyDrive/**/fatformer_crossdomain_weights_only.pth'
])

if not ckpt_path:
    print('[!] CẢNH BÁO: Chưa tìm thấy fatformer_crossdomain_weights_only.pth trên Google Drive!')
    print('    Hãy tải file fatformer_crossdomain_weights_only.pth lên Drive hoặc kéo thả vào thư mục /content/ trên Colab.')
else:
    print(f'[1] Checkpoint Khóa Sổ: {ckpt_path} ({os.path.getsize(ckpt_path)/(1024**3):.2f} GB)')

# 2. Tìm CLIP ViT-L/14 Backbone
clip_path = find_file([
    f'{DRIVE_ROOT}/**/ViT-L-14.pt',
    '/content/**/ViT-L-14.pt',
    '/content/drive/MyDrive/**/ViT-L-14.pt'
])
print(f'[2] CLIP ViT-L/14 Backbone: {clip_path}')



## ⚡ BƯỚC 3: GIẢI NÉN DỮ LIỆU TỪ GOOGLE DRIVE SANG SSD NVME COLAB
> Trích xuất dữ liệu chống nghẽn I/O từ Drive sang SSD NVMe Colab:
> - **Clean (18 subsets)**: `/content/dataset_local/test_clean`
> - **Degraded (6 variants)**: `/content/dataset_local/test_degraded`

In [ ]:
# ==============================================================================
# BƯỚC 3: TRÍCH XUẤT DỮ LIỆU TỪ GOOGLE DRIVE SANG SSD NVME MÁY ẢO COLAB
# ==============================================================================
import tarfile

LOCAL_CLEAN_DIR = "/content/dataset_local/test_clean"
LOCAL_DEG_DIR = "/content/dataset_local/test_degraded"
os.makedirs(LOCAL_CLEAN_DIR, exist_ok=True)
os.makedirs("/content/dataset_local", exist_ok=True)

# 1. Tìm các tệp .tar trên Google Drive
tar_gans = find_file([
    f'{DRIVE_ROOT}/datasets/test_benchmark_gans.tar',
    f'{DRIVE_ROOT}/**/test_benchmark_gans.tar',
    '/content/drive/MyDrive/**/test_benchmark_gans.tar'
])

tar_diff = find_file([
    f'{DRIVE_ROOT}/datasets/test_benchmark_diffusion.tar',
    f'{DRIVE_ROOT}/**/test_benchmark_diffusion.tar',
    '/content/drive/MyDrive/**/test_benchmark_diffusion.tar'
])

tar_deg = find_file([
    f'{DRIVE_ROOT}/datasets/test_degraded.tar',
    f'{DRIVE_ROOT}/**/test_degraded.tar',
    '/content/drive/MyDrive/**/test_degraded.tar',
    '/content/test_degraded.tar'
])

print(f"[*] Tệp GANs tar: {tar_gans}")
print(f"[*] Tệp Diffusion tar: {tar_diff}")
print(f"[*] Tệp Degraded tar: {tar_deg}")

# 2. Giải nén sang SSD NVMe máy ảo Colab
def extract_tar_safely(tar_path, target_dir):
    if not tar_path or not os.path.exists(tar_path):
        print(f"[!] Bỏ qua: Không tìm thấy {tar_path}")
        return
    print(f"[*] Đang sao chép và giải nén {os.path.basename(tar_path)} sang {target_dir}...")
    with tarfile.open(tar_path, 'r') as tf:
        tf.extractall(target_dir)
    print(f"[✓] Đã giải nén hoàn tất: {os.path.basename(tar_path)}")

if tar_gans:
    extract_tar_safely(tar_gans, LOCAL_CLEAN_DIR)
if tar_diff:
    extract_tar_safely(tar_diff, LOCAL_CLEAN_DIR)
if tar_deg:
    extract_tar_safely(tar_deg, "/content/dataset_local")

# 3. Kiểm tra số lượng tập test đã sẵn sàng
clean_subsets = [f for f in os.listdir(LOCAL_CLEAN_DIR) if os.path.isdir(os.path.join(LOCAL_CLEAN_DIR, f))] if os.path.exists(LOCAL_CLEAN_DIR) else []
deg_variants = [f for f in os.listdir(LOCAL_DEG_DIR) if os.path.isdir(os.path.join(LOCAL_DEG_DIR, f))] if os.path.exists(LOCAL_DEG_DIR) else []

print(f"\n[✓] Clean Subsets ({len(clean_subsets)}): {clean_subsets}")
print(f"[✓] Degraded Variants ({len(deg_variants)}): {deg_variants}")



## 🚀 BƯỚC 4: THỰC THI KIỂM CHUẨN KHÓA SỔ 2 CHIỀU TRÊN GPU (LOCKED BENCHMARK)
> Cố định $T=1.0811, T^*=0.3700$, chạy suy luận trên toàn bộ ảnh test Clean & 6 biến thể Degraded với GPU, tính toán ma trận sụt giảm $\Delta_{\text{drop}}$ và khoảng tin cậy 95% Bootstrap ($B=1.000$).

In [ ]:
# ==============================================================================
# BƯỚC 4: CHẠY KIỂM CHUẨN KHÓA SỔ 2 CHIỀU TRÊN GPU (LOCKED BENCHMARK)
# ==============================================================================
import subprocess

OUTPUT_LOGS = "/content/output_task_6.6"
os.makedirs(OUTPUT_LOGS, exist_ok=True)

cmd = [
    sys.executable, "-u", f"{REPO_DIR}/tools/eval_locked_benchmark.py",
    "--checkpoint", ckpt_path or "",
    "--clip_path", clip_path or "ViT-L-14.pt",
    "--temperature", "1.0811",
    "--threshold", "0.3700",
    "--output_dir", OUTPUT_LOGS,
    "--batch_size", "32",
    "--num_workers", "2",
    "--n_bootstrap", "1000",
    "--device", "cuda"
]

has_clean = os.path.exists(LOCAL_CLEAN_DIR) and len(os.listdir(LOCAL_CLEAN_DIR)) > 0
has_deg = os.path.exists(LOCAL_DEG_DIR) and len(os.listdir(LOCAL_DEG_DIR)) > 0

if has_clean:
    cmd.extend(["--test_root", LOCAL_CLEAN_DIR])
if has_deg:
    cmd.extend(["--degraded_root", LOCAL_DEG_DIR, "--samples_per_degraded_subset", "500"])

if not has_clean and not has_deg:
    print("[!] CẢNH BÁO: Không có ảnh test thực tế -> Chạy chế độ Dummy")
    cmd.append("--test_dummy")

print("=" * 80)
print(f"[*] Lệnh thực thi: {' '.join(cmd)}")
print("=" * 80)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")
process.wait()

if process.returncode != 0:
    raise RuntimeError(f"Tiến trình kết thúc với mã lỗi: {process.returncode}")
else:
    print("\n[✓] HOÀN TẤT VÒNG ĐO ĐẠC KIỂM CHUẨN CHUNG CUỘC 2 CHIỀU!")



## 📊 BƯỚC 5: TỔNG HỢP KẾT QUẢ, VẼ BIỂU ĐỒ 3 BÊN & LƯU VỀ GOOGLE DRIVE 5TB
> Hiển thị bảng ma trận kết quả Clean + Degraded, tự động vẽ biểu đồ đối chiếu 3 bên (CVPR 2024 vs Plan A vs M4) và tự động đồng bộ sang Google Drive `logs/task_6.6/`.

In [ ]:
# ==============================================================================
# BƯỚC 5: BẢNG TỔNG HỢP ĐỐI CHIẾU 3 BÊN VÀ ĐỒNG BỘ GOOGLE DRIVE
# ==============================================================================
import pandas as pd
from IPython.display import display, Markdown

res_clean_csv = os.path.join(OUTPUT_LOGS, 'final_posthoc_benchmark_matrix.csv')
res_deg_csv = os.path.join(OUTPUT_LOGS, 'final_locked_degraded_matrix.csv')
res_json = os.path.join(OUTPUT_LOGS, 'final_locked_benchmark_report.json')

if os.path.exists(res_clean_csv):
    df_clean = pd.read_csv(res_clean_csv)
    display(Markdown('### 📊 1. KẾT QUẢ KIỂM CHUẨN KHÓA SỔ TRÊN 18 TẬP TEST CLEAN:'))
    display(df_clean)

if os.path.exists(res_deg_csv):
    df_deg = pd.read_csv(res_deg_csv)
    display(Markdown('### 📉 2. KẾT QUẢ KIỂM CHUẨN KHÓA SỔ TRÊN 6 BIẾN THỂ DEGRADED:'))
    display(df_deg)

# Vẽ biểu đồ 3 bên nếu có script
plot_script = f'{REPO_DIR}/tools/plot_three_way_benchmark.py'
if os.path.exists(plot_script):
    try:
        subprocess.run([sys.executable, plot_script], check=True)
        chart_p = f'{REPO_DIR}/DATASET/logs/task_6.6_three_way_comparison.png'
        if os.path.exists(chart_p):
            shutil.copy2(chart_p, os.path.join(OUTPUT_LOGS, 'task_6.6_three_way_comparison.png'))
            print('[✓] Đã tạo biểu đồ đối chiếu 3 bên: task_6.6_three_way_comparison.png')
    except Exception as e:
        print(f'[!] Cảnh báo vẽ biểu đồ: {e}')

# 1. Đóng gói toàn bộ kết quả vào tệp zip
export_zip = '/content/task_6.6_final_results.zip'
with zipfile.ZipFile(export_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_LOGS):
        for fn in files:
            fp = os.path.join(root, fn)
            arcname = os.path.relpath(fp, OUTPUT_LOGS)
            zf.write(fp, arcname)

# 2. Tự động lưu trữ trực tiếp vào Google Drive 5TB
drive_out_dir = os.path.join(DRIVE_ROOT, "logs", "task_6.6")
os.makedirs(drive_out_dir, exist_ok=True)
for item in os.listdir(OUTPUT_LOGS):
    s_item = os.path.join(OUTPUT_LOGS, item)
    d_item = os.path.join(drive_out_dir, item)
    if os.path.isdir(s_item):
        shutil.copytree(s_item, d_item, dirs_exist_ok=True)
    else:
        shutil.copy2(s_item, d_item)

shutil.copy2(export_zip, os.path.join(drive_out_dir, "task_6.6_final_results.zip"))
print(f"[✓] ĐÃ TỰ ĐỘNG LƯU TRỮ TOÀN BỘ KẾT QUẢ VÀO GOOGLE DRIVE: {drive_out_dir}")

# 3. Kích hoạt tải trực tiếp về máy tính nếu muốn
try:
    from google.colab import files
    files.download(export_zip)
    print("[✓] Đã kích hoạt tải tệp task_6.6_final_results.zip về máy!")
except Exception:
    pass

print('\n🎉 HOÀN TẤT TOÀN DIỆN VÒNG ĐO ĐẠC TASK 6.6 TRÊN GOOGLE COLAB PRO!')

